# 🎸 BOSS IR2 - Parameter Mapping

## Protocollo Scoperto ✅

```
F0 41 10 01 05 09 12 20 00 00 XX YY ZZ F7
   │  │  └─┴─┴─ Model ID: 01 05 09
   │  └──────── Device ID: 10
   └─────────── Roland: 41
                  └──────── DT1 (Data Set): 12
                        └─┴─┴─┴─ Address: 20 00 00 XX
                                 └─ Value: YY
                                    └─ Checksum: ZZ
```

**Obiettivo**: Mappare ogni parametro fisico → indirizzo memoria

In [ ]:
import mido
import time
import json
import os
from datetime import datetime
from IPython.display import clear_output

os.makedirs("captures", exist_ok=True)

# Auto-detect
IR2_INPUT = None
for name in mido.get_input_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_INPUT = name
        break

print(f"✅ Dispositivo: {IR2_INPUT}")

---
## 🎛️ Cattura Parametro Singolo

**Istruzioni:**
1. Modifica `PARAMETER_NAME` con il nome del parametro
2. Esegui la cella
3. Muovi **SOLO** quel parametro da MIN a MAX
4. Aspetta che finisca automaticamente

In [ ]:
# ==================== CONFIGURAZIONE ====================
PARAMETER_NAME = "VOLUME"  # ⬅️ MODIFICA QUI!
DURATION = 15  # secondi
# ========================================================

In [ ]:
# 🎧 CATTURA ATTIVA

print(f"🎛️ Parametro: {PARAMETER_NAME}")
print(f"⏱️ Durata: {DURATION} secondi")
print("="*60)
print(f"👉 Muovi SOLO '{PARAMETER_NAME}' da MIN a MAX!")
print("="*60)
print()

captured = []
start_time = time.time()
last_value = None

with mido.open_input(IR2_INPUT) as port:
    while time.time() - start_time < DURATION:
        remaining = int(DURATION - (time.time() - start_time))
        
        msg = port.poll()
        
        if msg and msg.type == 'sysex':
            ts = datetime.now().strftime("%H:%M:%S.%f")[:-3]
            
            entry = {
                "time": ts,
                "data": list(msg.data),
                "hex": " ".join(f"{b:02X}" for b in msg.data)
            }
            
            # Parse Roland
            if len(msg.data) >= 11 and msg.data[0] == 0x41:
                entry["parsed"] = {
                    "address": msg.data[6:10],
                    "value": msg.data[10]
                }
                
                value = msg.data[10]
                addr = " ".join(f"{b:02X}" for b in msg.data[6:10])
                
                # Mostra solo se valore cambia
                if value != last_value:
                    print(f"[{remaining:2d}s] Address: {addr} | Value: {value:3d} (0x{value:02X})")
                    last_value = value
            
            captured.append(entry)
        
        time.sleep(0.001)

print("\n" + "="*60)
print(f"✅ Catturati {len(captured)} messaggi")

In [ ]:
# 📊 ANALISI E SALVATAGGIO

if not captured:
    print("⚠️ Nessun messaggio catturato!")
else:
    # Estrai indirizzi unici
    addresses = set()
    values = []
    
    for m in captured:
        if "parsed" in m:
            addr = tuple(m["parsed"]["address"])
            addresses.add(addr)
            values.append(m["parsed"]["value"])
    
    print(f"📋 ANALISI '{PARAMETER_NAME}'")
    print("="*60)
    print(f"Messaggi SysEx: {len(captured)}")
    print(f"Indirizzi unici: {len(addresses)}")
    
    for addr in sorted(addresses):
        addr_hex = " ".join(f"{b:02X}" for b in addr)
        print(f"  • {addr_hex}")
    
    if values:
        print(f"\nRange valori:")
        print(f"  MIN: {min(values):3d} (0x{min(values):02X})")
        print(f"  MAX: {max(values):3d} (0x{max(values):02X})")
        print(f"  Valori unici: {len(set(values))}")
    
    # Salva
    filename = f"captures/{PARAMETER_NAME}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    with open(filename, 'w') as f:
        json.dump({
            "parameter": PARAMETER_NAME,
            "duration": DURATION,
            "timestamp": datetime.now().isoformat(),
            "messages": captured,
            "summary": {
                "addresses": [list(a) for a in addresses],
                "min_value": min(values) if values else None,
                "max_value": max(values) if values else None,
                "unique_values": len(set(values)) if values else 0
            }
        }, f, indent=2)
    
    print(f"\n💾 Salvato: {filename}")

---
## 📋 Mappa Parametri Completa

Carica tutti i file catturati e crea la mappa finale:

In [ ]:
import glob

param_map = {}

for filepath in sorted(glob.glob("captures/*.json")):
    with open(filepath) as f:
        data = json.load(f)
    
    param = data.get("parameter", "unknown")
    summary = data.get("summary", {})
    
    if summary.get("addresses"):
        addr = summary["addresses"][0]  # Prendi il primo (di solito è unico)
        param_map[param] = {
            "address": addr,
            "address_hex": " ".join(f"{b:02X}" for b in addr),
            "min": summary.get("min_value"),
            "max": summary.get("max_value"),
            "file": filepath
        }

print("🎛️ BOSS IR-2 PARAMETER MAP")
print("="*70)
print(f"{'Parametro':<20} {'Indirizzo':<15} {'Range':<15} {'File'}")
print("-"*70)

for param in sorted(param_map.keys()):
    info = param_map[param]
    range_str = f"{info['min']}-{info['max']}" if info['min'] is not None else "?"
    print(f"{param:<20} {info['address_hex']:<15} {range_str:<15} {os.path.basename(info['file'])}")

# Salva mappa
with open("parameter_map.json", 'w') as f:
    json.dump(param_map, f, indent=2)

print(f"\n💾 Mappa salvata in: parameter_map.json")

---
## 📝 Checklist Parametri

Torna alla **cella 2** e testa questi parametri uno alla volta:

- [ ] VOLUME
- [ ] LOW_EQ
- [ ] MID_EQ
- [ ] HIGH_EQ
- [ ] CABINET_SELECT
- [ ] PRESET_1
- [ ] PRESET_2
- [ ] PRESET_3
- [ ] (altri...)